# 07 Transformer 三协议实验（论文主结果）

目标：运行 Transformer 在 `same_condition` / `lobo` / `loco` 三协议下的全量评估，并自动保存结果与图表，便于论文写作。

## 1) 环境与依赖

- 模型：`ConditionAwareTransformer`
- 指标：RMSE / MAE / MAPE
- 输出：逐 split 结果、协议汇总结果、统一图表、论文表格

In [1]:
import os
import sys
import json
import random
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager

PROJECT_ROOT = r"C:\\Users\\PLUTO\\Desktop\\battery-rul"
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.Transformer.data import (
    load_battery_raw_parameters,
    default_condition_map,
    build_protocol_splits,
    prepare_condition_aware_dataloaders,
)
from src.Transformer.train import train_model, predict, compute_regression_metrics
from src.Transformer.models import ConditionAwareTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

def setup_cn_font():
    candidates = [
        'SimSun', 'Songti SC', 'NSimSun', 'Microsoft YaHei',
        'SimHei', 'Noto Sans CJK SC', 'Arial Unicode MS', 'DejaVu Sans'
    ]
    available = {f.name for f in font_manager.fontManager.ttflist}
    selected = next((name for name in candidates if name in available), 'DejaVu Sans')
    mpl.rcParams['font.family'] = selected
    mpl.rcParams['axes.unicode_minus'] = False
    return selected

font_name = setup_cn_font()
print("Plot Font:", font_name)

Device: cuda
Plot Font: SimSun


In [2]:
# 加载全部可用电池（按 condition_map 过滤）
PROCESSED_ROOT = os.path.join(PROJECT_ROOT, "data\\processed")
condition_map = default_condition_map()
target_ids = set(condition_map.keys())

battery_dict = {}
for group_name in sorted(os.listdir(PROCESSED_ROOT)):
    group_path = os.path.join(PROCESSED_ROOT, group_name)
    if not os.path.isdir(group_path):
        continue

    pkl_files = [f for f in os.listdir(group_path) if f.endswith(".pkl")]
    group_ids = [os.path.splitext(f)[0] for f in pkl_files]
    group_ids = [bid for bid in group_ids if bid in target_ids]
    if not group_ids:
        continue

    group_data = load_battery_raw_parameters(group_path, group_ids)
    battery_dict.update(group_data)

available_ids = sorted([bid for bid in battery_dict if bid in condition_map])
protocols = build_protocol_splits(available_ids, condition_map)

print("总电池数:", len(available_ids))
print("same_condition splits:", len(protocols['same_condition']))
print("lobo splits:", len(protocols['lobo']))
print("loco splits:", len(protocols['loco']))

总电池数: 34
same_condition splits: 34
lobo splits: 34
loco splits: 6


## 2) 三协议全量评估（Transformer）

说明：默认 `FULL_EVAL=True` 运行全量。若仅做快速检查，可改为 `False` 并设置 `QUICK_SPLITS`。

In [3]:
SEQ_LENGTH = 10
BATCH_SIZE = 32

TRAIN_KWARGS = {"epochs": 25, "lr": 1e-3}
MODEL_KWARGS = {
    "input_dim": 5,
    "d_model": 64,
    "nhead": 4,
    "num_layers": 2,
    "dim_feedforward": 128,
    "dropout": 0.1,
    "max_seq_len": 256,
}

FULL_EVAL = True
QUICK_SPLITS = 3

def run_transformer_for_protocol(protocol_name, split_list):
    rows = []
    use_splits = split_list if FULL_EVAL else split_list[:QUICK_SPLITS]
    print(f"\n===== 协议: {protocol_name} | splits: {len(use_splits)} =====")

    for split_id, split in enumerate(use_splits):
        train_ids = split['train_ids']
        test_ids = split['test_ids']

        for test_id in test_ids:
            try:
                train_loader, test_loader, scaler_y, test_actual_capacity = prepare_condition_aware_dataloaders(
                    battery_dict=battery_dict,
                    train_ids=train_ids,
                    test_id=test_id,
                    condition_map=condition_map,
                    seq_length=SEQ_LENGTH,
                    batch_size=BATCH_SIZE,
                )
            except ValueError as error:
                print(f"[Skip] protocol={protocol_name} split={split_id} test={test_id} reason={error}")
                continue

            model = ConditionAwareTransformer(**MODEL_KWARGS)
            _ = train_model(
                model=model,
                train_loader=train_loader,
                device=device,
                **TRAIN_KWARGS,
            )

            preds = predict(model=model, test_loader=test_loader, scaler_y=scaler_y, device=device)
            y_true = test_actual_capacity[SEQ_LENGTH:SEQ_LENGTH + len(preds)]
            metrics = compute_regression_metrics(y_true, preds)

            rows.append({
                "protocol": protocol_name,
                "split_id": split_id,
                "group": split.get('group', 'N/A'),
                "test_id": test_id,
                "num_train_batteries": len(train_ids),
                "num_points": int(len(preds)),
                "RMSE": metrics['RMSE'],
                "MAE": metrics['MAE'],
                "MAPE": metrics['MAPE'],
            })

    return pd.DataFrame(rows)

all_parts = []
for protocol_name in ['same_condition', 'lobo', 'loco']:
    df_part = run_transformer_for_protocol(protocol_name, protocols[protocol_name])
    print(f"{protocol_name} 有效记录数: {len(df_part)}")
    all_parts.append(df_part)

df_transformer_all_protocols = pd.concat(all_parts, ignore_index=True) if all_parts else pd.DataFrame()
print("\n=== Transformer 三协议结果总览 ===")
print("总记录数:", len(df_transformer_all_protocols))
display(df_transformer_all_protocols.head())

if not df_transformer_all_protocols.empty:
    summary_protocol_transformer = (
        df_transformer_all_protocols.groupby('protocol')[['RMSE', 'MAE', 'MAPE']]
        .agg(['mean', 'std', 'count'])
    )
    print("\n=== 协议汇总（mean/std/count）===")
    display(summary_protocol_transformer)

    # 保存结果（逐split + 协议汇总）
    save_dir = os.path.join(PROJECT_ROOT, 'results', 'transformer_baseline')
    os.makedirs(save_dir, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')

    split_csv = os.path.join(save_dir, f'transformer_three_protocols_split_results_{ts}.csv')
    summary_csv = os.path.join(save_dir, f'transformer_three_protocols_summary_{ts}.csv')

    df_transformer_all_protocols.to_csv(split_csv, index=False, encoding='utf-8-sig')
    summary_protocol_transformer.reset_index().to_csv(summary_csv, index=False, encoding='utf-8-sig')

    print('\n已保存:')
    print(split_csv)
    print(summary_csv)


===== 协议: same_condition | splits: 34 =====
[Skip] protocol=same_condition split=29 test=B0052 reason=测试电池 B0052 样本不足，无法构建窗口
same_condition 有效记录数: 33

===== 协议: lobo | splits: 34 =====
[Skip] protocol=lobo split=29 test=B0052 reason=测试电池 B0052 样本不足，无法构建窗口
lobo 有效记录数: 33

===== 协议: loco | splits: 6 =====
[Skip] protocol=loco split=5 test=B0052 reason=测试电池 B0052 样本不足，无法构建窗口
loco 有效记录数: 33

=== Transformer 三协议结果总览 ===
总记录数: 99


,protocol,split_id,group,test_id,num_train_batteries,num_points,RMSE,MAE,MAPE
0,same_condition,0,T24_I2,B0005,4,158,0.092332,0.082833,5.688851
1,same_condition,1,T24_I2,B0006,4,158,0.064038,0.048790,3.245656
2,same_condition,2,T24_I2,B0007,4,158,0.032544,0.027078,1.631308
3,same_condition,3,T24_I2,B0018,4,122,0.063807,0.051496,3.276653
4,same_condition,4,T24_I2,B0036,4,187,0.094648,0.061813,3.573389



=== 协议汇总（mean/std/count）===


RMSE                       MAE                       MAPE  \
                    mean       std count      mean       std count       mean   
protocol                                                                        
lobo            0.196624  0.163720    33  0.167089  0.160335    33  60.226599   
loco            0.446154  0.236906    33  0.421520  0.237855    33  83.825587   
same_condition  0.187735  0.189875    33  0.158015  0.183677    33  60.639819   

                                  
                       std count  
protocol                          
lobo            169.360502    33  
loco            164.213851    33  
same_condition  174.991627    33


已保存:
C:\\Users\\PLUTO\\Desktop\\battery-rul\results\transformer_baseline\transformer_three_protocols_split_results_20260324_160427.csv
C:\\Users\\PLUTO\\Desktop\\battery-rul\results\transformer_baseline\transformer_three_protocols_summary_20260324_160427.csv


## 3) 统一图表（论文可直接使用）

- 每个协议：1 张误差分布图（MAE / RMSE / MAPE）
- 全协议：1 张对比箱线图
- 自动保存 PNG 到 `results/transformer_baseline/figures`

In [ ]:
if 'df_transformer_all_protocols' not in globals() or df_transformer_all_protocols.empty:
    print('df_transformer_all_protocols 为空，请先运行三协议主单元。')
else:
    df_show = df_transformer_all_protocols.copy()
    metrics = ['MAE', 'RMSE', 'MAPE']
    protocol_order = ['same_condition', 'lobo', 'loco']

    fig_dir = os.path.join(PROJECT_ROOT, 'results', 'transformer_baseline', 'figures')
    os.makedirs(fig_dir, exist_ok=True)
    fig_ts = datetime.now().strftime('%Y%m%d_%H%M%S')

    # 每协议一张误差分布图
    for protocol_name in protocol_order:
        part = df_show[df_show['protocol'] == protocol_name].copy()
        if part.empty:
            print(f'[{protocol_name}] 无数据，跳过。')
            continue

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        fig.suptitle(f'Transformer - {protocol_name} 协议误差分布', fontsize=13)

        for ax, metric in zip(axes, metrics):
            values = part[metric].dropna().values
            if len(values) == 0:
                ax.set_title(f'{metric}（无数据）')
                ax.axis('off')
                continue

            ax.hist(values, bins=12, alpha=0.75, edgecolor='black')
            mean_v = float(np.mean(values))
            med_v = float(np.median(values))
            ax.axvline(mean_v, linestyle='--', linewidth=1.8, label=f'均值={mean_v:.3f}')
            ax.axvline(med_v, linestyle=':', linewidth=1.8, label=f'中位数={med_v:.3f}')
            ax.set_title(metric)
            ax.set_xlabel('误差值')
            ax.set_ylabel('频数')
            ax.grid(alpha=0.3)
            ax.legend(fontsize=8)

        plt.tight_layout()
        out_path = os.path.join(fig_dir, f'transformer_{protocol_name}_error_dist_{fig_ts}.png')
        plt.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.show()
        print('已保存图:', out_path)

    # 三协议对比箱线图
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    fig.suptitle('Transformer 三协议误差对比图（箱线图）', fontsize=13)

    for ax, metric in zip(axes, metrics):
        data = [df_show.loc[df_show['protocol'] == p, metric].dropna().values for p in protocol_order]
        ax.boxplot(data, tick_labels=protocol_order, showfliers=False)
        ax.set_title(metric)
        ax.set_xlabel('协议')
        ax.set_ylabel('误差')
        ax.grid(alpha=0.3)

    plt.tight_layout()
    compare_path = os.path.join(fig_dir, f'transformer_three_protocols_boxplot_{fig_ts}.png')
    plt.savefig(compare_path, dpi=300, bbox_inches='tight')
    plt.show()
    print('已保存图:', compare_path)

## 4) 论文表格生成（mean±std）

该单元会输出可直接放入论文的汇总表，并保存为 CSV。

In [ ]:
if 'df_transformer_all_protocols' not in globals() or df_transformer_all_protocols.empty:
    print('df_transformer_all_protocols 为空，请先运行三协议主单元。')
else:
    rows = []
    for p in ['same_condition', 'lobo', 'loco']:
        part = df_transformer_all_protocols[df_transformer_all_protocols['protocol'] == p]
        if part.empty:
            continue
        rows.append({
            '协议': p,
            'RMSE (Ah)': f"{part['RMSE'].mean():.4f} ± {part['RMSE'].std():.4f}",
            'MAE (Ah)': f"{part['MAE'].mean():.4f} ± {part['MAE'].std():.4f}",
            'MAPE (%)': f"{part['MAPE'].mean():.4f} ± {part['MAPE'].std():.4f}",
            '样本数': int(len(part)),
        })

    table_df = pd.DataFrame(rows)
    print('=== 表4-xx Transformer 三协议总体评估结果（mean±std）===')
    display(table_df)

    save_dir = os.path.join(PROJECT_ROOT, 'results', 'transformer_baseline')
    os.makedirs(save_dir, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    table_path = os.path.join(save_dir, f'transformer_three_protocols_table_mean_std_{ts}.csv')
    table_df.to_csv(table_path, index=False, encoding='utf-8-sig')
    print('已保存表格:', table_path)